# Práctica: puntuaciones Metacritic y modelos de regresión (juegos PC)

**Audiencia:** estudiantes con interés en desarrollo de videojuegos y datos.  
**Herramientas:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`.  
**Kernel recomendado:** `Python (PythonExercises venv)`.

---

## Cómo está organizado este cuaderno

| Tipo de celda | Qué haces tú |
|---------------|----------------|
| **Markdown** con título **«Muestra / referencia (solo lectura)»** | **No es tu entrega.** Lee el patrón de código; los ejemplos suelen ser de **otro dominio** (tiendas, inventario) o muestran **sintaxis genérica**. |
| **Código** con encabezado **`# === Tu código — Paso X ===`** | **Aquí escribes y ejecutas** todo lo necesario para ese paso. Esas celdas empiezan **vacías** (o casi). |

**Orden:** ejecuta los pasos en secuencia (2 → 3 → 4 …). Si saltas un paso, las variables (`df`, `df_filtrado`, `X_train`, etc.) no existirán y fallará el siguiente.

---

## 1. Dataset: origen y objetivo

**Fuente:** `metacritic_games.csv` (*basic-dataset*, prasertcbs, GitHub).

**Objetivo:** regresión para estimar **Metascore** a partir de **año** y **conteos de críticos**; comparar regresión lineal y Random Forest.

**Importante:** correlación no implica causalidad; Metacritic es un agregado editorial.

## 1.1 Significado de cada columna y relaciones

| Columna | Significado breve |
|---------|---------------------|
| **game** | Nombre del juego (identificador legible). |
| **platform** | PC, PS4, 3DS, etc. |
| **developer** / **genre** / **number_players** / **rating** | Metadatos; útiles para filtros o EDA, no entran en el `X` mínimo de esta práctica. |
| **release_date** | Texto tipo `Apr 18, 2011` → derivar **`release_year`**. |
| **positive_critics**, **neutral_critics**, **negative_critics** | Conteos de reseñas de críticos en el agregado. |
| **positive_users**, **neutral_users**, **negative_users**, **user_score** | No se usan en el modelo base de esta práctica. |
| **metascore** | **Objetivo `y`** (0–100 típico). |

**Relaciones:** `game`+`platform` identifican una fila; el **metascore** resume la recepción crítica, por eso suele correlacionar con los conteos; **release_year** sitúa el juego en el tiempo (mercado y expectativas).

**`X` en esta práctica:** `release_year`, `positive_critics`, `neutral_critics`, `negative_critics`.

---
## Paso 2 — Cargar el dataset

**Tu tarea en la celda de código de abajo:** importar `pandas`, definir la URL del CSV (o ruta local), cargar en `df` y explorar.

**Qué deberías ver al ejecutar bien este paso:**

- `df.shape` con **miles de filas** y alrededor de **17 columnas**.
- `head()` con columnas como `game`, `platform`, `metascore`, `release_date`, etc.
- Si el resultado no coincide, revisa la URL o el separador del CSV.

### Muestra / referencia (solo lectura)

Ejemplo en **otro dominio** (inventario de una tienda). **No copies literalmente**; adapta nombres y URL a Metacritic:

```python
import pandas as pd
url = "https://ejemplo.com/productos.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()
```

**URL de este práctico (para tu celda ejecutable):**  
`https://raw.githubusercontent.com/prasertcbs/basic-dataset/master/metacritic_games.csv`

In [ ]:
# === Tu código — Paso 2: carga y exploración ===



---
## Paso 3 — Nombres de columnas y `release_year`

**Tu tarea:** quitar espacios en encabezados; crear `release_year` desde `release_date` con `pd.to_datetime(..., errors="coerce").dt.year`.

**Qué esperar:**

- Tras `strip`, nombres como `user_score` sin espacio final.
- `release_year` con valores como 2011, 2012…; `release_year.isna().sum()` puede ser **> 0** si alguna fecha es rara — anótalo.
- El **formato** de fechas en este archivo es mes abreviado en inglés: `%b %d, %Y` (ej. `Apr 18, 2011`).

### Muestra / referencia (solo lectura)

```python
# Otro dataset: columna fecha ISO
df.columns = df.columns.str.strip()
df["anio"] = pd.to_datetime(df["fecha_iso"], format="%Y-%m-%d", errors="coerce").dt.year
```

En **tu** caso la columna se llama `release_date` y el `format` es `'%b %d, %Y'`.

In [ ]:
# === Tu código — Paso 3: strip + release_year ===



---
## Paso 4 — Filtrar filas y preparar `df_filtrado`

**Tu tarea:**

1. Define **dos criterios** (variables) que documentarás: por ejemplo plataforma y año mínimo.
2. Construye una **máscara booleana** y `df_filtrado = df.loc[mascara].copy()`.
3. Elimina filas sin `metascore` o sin `release_year` válido (`dropna`).

**Qué esperar y cómo interpretarlo:**

- Imprime **antes → después** el número de filas. La caída es **normal**: solo quedan juegos que cumplen **todos** los criterios.
- En un comentario en tu código, escribe **una frase** del tipo: *"Conservo solo juegos de plataforma X lanzados desde el año Y con Metascore conocido."*
- Si el subconjunto queda muy pequeño, relaja el año mínimo o revisa el filtro.

### Muestra / referencia (solo lectura)

```python
# Ejemplo genérico: productos de una categoría y precio conocido
categoria_objetivo = "A"
precio_min = 10
m = (df["cat"] == categoria_objetivo) & (df["precio"] >= precio_min)
sub = df.loc[m].copy()
sub = sub.dropna(subset=["precio", "stock"])
print(len(df), "->", len(sub))
```

In [ ]:
# === Tu código — Paso 4: filtro + dropna → df_filtrado ===



---
## Paso 5 — Definir `X`, `y` y `juegos`

**Tu tarea:**

- `y` = `metascore`
- `X` = solo `release_year`, `positive_critics`, `neutral_critics`, `negative_critics`
- `juegos` = columna `game` (mismo índice que cada fila de `X`)

**Qué esperar:** `X.shape[0]`, `len(y)` y `len(juegos)` deben ser **iguales**. Si no, revisa índices o subconjuntos.

### Muestra / referencia (solo lectura)

```python
cols = ["a", "b"]
X = sub[cols]
y = sub["objetivo"]
ids = sub["id_producto"]
```

In [ ]:
# === Tu código — Paso 5: X, y, juegos ===



---
## Paso 6 — `train_test_split` con `juegos` alineado

**Tu tarea:** partir `X`, `y` y `juegos` con el mismo `test_size` y `random_state` (ej. `0.2` y `42`).

**Qué esperar:**

- Aproximadamente **80%** de filas en entrenamiento y **20%** en prueba (el exacto depende del tamaño y de `test_size`).
- Los índices de `X_train` coinciden con los de `y_train` y `juegos_train` (misma fila = mismo juego).

### Muestra / referencia (solo lectura)

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Para alinear también nombres: añade un tercer array en train_test_split(...)
```

In [ ]:
# === Tu código — Paso 6: train_test_split(X, y, juegos, ...) ===



---
## Paso 7 — Interpretar el filtro, el split y las **primeras filas** de train / test

**Tu tarea en la celda de código:**

1. **Resume el filtro en texto** con `print`: qué valores usaste para plataforma y año mínimo (o los criterios que elegiste). Así queda claro **qué población** estás modelando.
2. **Tres primeros juegos del entrenamiento:** muestra `juegos_train` (por ejemplo `.head(3)` o `.iloc[:3]`) **junto** con sus `y_train` y, si quieres, las filas correspondientes de `X_train`.
   - **Qué significa:** esos títulos y Metascores son los que el modelo **sí verá** al hacer `.fit()` — son la base del aprendizaje.
3. **Tres primeros juegos del conjunto de prueba:** igual con `juegos_test`, `y_test` y opcionalmente `X_test`.
   - **Qué significa:** el modelo **no** debe haber usado esas filas en el entrenamiento; sirven solo para **medir** qué tan bien generaliza.
4. (Opcional) Comprueba que un juego de `juegos_test` **no** aparece en `juegos_train` (por ejemplo el primero del test frente a `set(juegos_train)`).

**Qué esperar:** nombres reales de juegos PC (si filtraste PC); Metascores entre ~30 y ~100 típicamente; las listas train y test **no** deben solaparse en los mismos índices de split.

### Muestra / referencia (solo lectura)

```python
# Ilustración con listas ficticias (no es Metacritic)
print("Filtro aplicado: solo registros con region='EU'")
nombres_train = pd.Series(["Cliente A", "Cliente B", "Cliente C"])
valor_train = pd.Series([100, 200, 150])
for i in range(3):
    print(nombres_train.iloc[i], "-> objetivo:", valor_train.iloc[i])
```

In [ ]:
# === Tu código — Paso 7: explicar filtro + 3 train + 3 test + interpretación ===



---
## Paso 8 — Distribución de `y_train` frente a `y_test`

**Tu tarea:** `describe()` o media/desv. en ambos; opcional histogramas.

**Qué esperar:** medias y rangos **parecidos** sugieren un split equilibrado para Metascore; grandes diferencias indican que train y test no representan lo mismo (interpreta con cuidado las métricas finales).

### Muestra / referencia (solo lectura)

```python
print(y_train.describe())
print(y_test.describe())
```

In [ ]:
# === Tu código — Paso 8: estadísticos y/o histogramas y_train vs y_test ===



---
## Paso 9 — Títulos de seguimiento (¿train o test?)

**Tu tarea:** lista de 2–4 nombres **exactos** como en `game`; imprime si caen en train, test o ninguno.

**Qué esperar:** depende de `random_state` y del filtro; *Portal 2* puede estar solo en train, solo en test o fuera si no pasó el filtro.

### Muestra / referencia (solo lectura)

```python
candidatos = ["Item X", "Item Y"]
for c in candidatos:
    print(c, "train:", c in set(ids_train), "test:", c in set(ids_test))
```

In [ ]:
# === Tu código — Paso 9: titulos_seguimiento ===



---
## Paso 10 — Entrenar regresión lineal y Random Forest

**Tu tarea:** `fit` en train; predicciones en train y test para ambos modelos.

### Muestra / referencia (solo lectura)

```python
from sklearn.linear_model import LinearRegression
m = LinearRegression()
m.fit(X_train, y_train)
pred_test = m.predict(X_test)
```

In [ ]:
# === Tu código — Paso 10: LinearRegression + RandomForestRegressor + predict ===



---
## Paso 11 — MSE y R² (train y test)

**Tu tarea:** `mean_squared_error`, `r2_score` para LR y RF en ambos conjuntos; una frase de conclusión.

### Muestra / referencia (solo lectura)

```python
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(y_test, pred_test)
r2 = r2_score(y_test, pred_test)
```

In [ ]:
# === Tu código — Paso 11: métricas ===



---
## Paso 12 — Gráficos: qué mirar y cómo interpretarlos

**Eje X:** Metascore real en test. **Eje Y:** predicho. **Línea y = x:** predicción perfecta.

**Interpretación:** puntos por encima = sobrestimación; por debajo = subestimación; nube lejana de la diagonal = más error.

**Tu tarea:** scatter (LR y RF) + línea identidad; 2–3 frases de interpretación **en una celda markdown nueva** o en comentarios al final de tu código.

### Muestra / referencia (solo lectura)

```python
import matplotlib.pyplot as plt
plt.scatter(y_real, y_pred, alpha=0.5)
m = min(y_real.min(), y_pred.min()); M = max(y_real.max(), y_pred.max())
plt.plot([m, M], [m, M], "k--")
plt.xlabel("Real"); plt.ylabel("Predicho")
plt.show()
```

### Tu interpretación del gráfico (Paso 12 — texto libre)

*(Escribe aquí 2–3 frases después de generar el scatter: ¿los puntos siguen la diagonal? ¿LR o RF se acerca más? ¿En qué rango de Metascore ves más error?)*


In [ ]:
# === Tu código — Paso 12: gráficos real vs predicho ===



---
## Paso 13 — Regresión lineal vs Random Forest en este estudio

| | Lineal | Random Forest |
|-|--------|----------------|
| Forma | Combinación lineal de las 4 entradas. | Promedio de muchos árboles (reglas). |
| Interpretación | Coeficientes por variable. | `feature_importances_`; más “caja oscura”. |
| Riesgo | Suele subajustar si la relación es muy no lineal. | Puede sobreajustar con árboles muy profundos. |
| Decisión | Compara **R² y MSE en test**, no solo en train. |

**Tu tarea:** en el cuaderno o en clase, explica con **tus números** cuál modelo prefieres en test y por qué.

---

## Lista de comprobación

- [ ] Cada **Paso N** tiene código solo en la celda «Tu código — Paso N»
- [ ] Paso 7: filtro explicado + 3 train + 3 test interpretados
- [ ] Paso 8–12 completados
- [ ] Reflexión LR vs RF con tus métricas

**Datos:** `metacritic_games.csv` — *basic-dataset* (prasertcbs), GitHub.